# Definition et Calcul des KPI

## Optimisation du Reseau de Services Publics - Togo Datalab

**Objectif:** Definir et calculer les indicateurs cles de performance pour le pilotage du service.

---

### Table des matieres
1. Configuration et imports
2. Chargement des donnees nettoyees
3. Definition des KPI
4. KPI Performance operationnelle
5. KPI Accessibilite territoriale
6. KPI Qualite de service
7. KPI Efficience
8. Tableau de synthese
9. Export et recommendations

## 1. Configuration et imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Chemins
DATA_PATH = Path.cwd().parent / 'data' / 'processed'
OUTPUT_PATH = Path.cwd().parent / 'outputs' / 'exports'

# Dictionnaire pour stocker les resultats
kpi_results = {}

print("Configuration terminee.")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Chargement des donnees nettoyees

In [ ]:
print("Chargement des donnees nettoyees...")
print("-" * 50)

df_demandes = pd.read_csv(DATA_PATH / 'demandes_clean.csv')
df_centres = pd.read_csv(DATA_PATH / 'centres_clean.csv')
df_logs = pd.read_csv(DATA_PATH / 'logs_clean.csv')
df_socioeco = pd.read_csv(DATA_PATH / 'socioeco_clean.csv')
df_communes = pd.read_csv(DATA_PATH / 'communes_clean.csv')

print(f"Demandes: {len(df_demandes)} lignes")
print(f"Centres: {len(df_centres)} lignes")
print(f"Logs: {len(df_logs)} lignes")
print(f"Socioeco: {len(df_socioeco)} lignes")
print(f"Communes: {len(df_communes)} lignes")
print("\n[OK] Donnees chargees avec succes")

## 3. Definition des KPI

### Tableau de definition structuree

Les KPI sont organises en 4 categories strategiques correspondant aux objectifs de l'administration publique.

In [ ]:
# Definition des KPI avec leurs caracteristiques
KPI_DEFINITIONS = [
    {
        'id': 'KPI_01',
        'nom': 'Delai Moyen de Traitement',
        'categorie': 'Performance operationnelle',
        'objectif': 'Mesurer l\'efficacite du processus de traitement des demandes',
        'description': 'Duree moyenne entre le depot d\'une demande et sa finalisation',
        'regle_calcul': 'Somme(delais) / Nombre(demandes)',
        'unite': 'jours',
        'seuil_vert': 14,
        'seuil_orange': 21,
        'seuil_rouge': 30,
        'tendance_positive': 'baisse'
    },
    {
        'id': 'KPI_02',
        'nom': 'Taux Utilisation Capacite',
        'categorie': 'Performance operationnelle',
        'objectif': 'Optimiser l\'allocation des ressources humaines et materielles',
        'description': 'Ratio entre le volume effectivement traite et la capacite theorique',
        'regle_calcul': '(Traite par jour / Capacite) x 100',
        'unite': '%',
        'seuil_vert': 85,
        'seuil_orange': 70,
        'seuil_rouge': 50,
        'tendance_positive': 'hausse'
    },
    {
        'id': 'KPI_03',
        'nom': 'Ratio Population par Centre',
        'categorie': 'Accessibilite territoriale',
        'objectif': 'Evaluer l\'equite de distribution des centres sur le territoire',
        'description': 'Nombre moyen d\'habitants par centre de service actif',
        'regle_calcul': 'Population totale / Nombre de centres actifs',
        'unite': 'habitants/centre',
        'seuil_vert': 50000,
        'seuil_orange': 80000,
        'seuil_rouge': 100000,
        'tendance_positive': 'baisse'
    },
    {
        'id': 'KPI_04',
        'nom': 'Taux Couverture Communale',
        'categorie': 'Accessibilite territoriale',
        'objectif': 'Mesurer la presence territoriale du service public',
        'description': 'Pourcentage de communes disposant d\'au moins un centre actif',
        'regle_calcul': '(Communes avec centre / Total communes) x 100',
        'unite': '%',
        'seuil_vert': 80,
        'seuil_orange': 60,
        'seuil_rouge': 40,
        'tendance_positive': 'hausse'
    },
    {
        'id': 'KPI_05',
        'nom': 'Taux de Rejet',
        'categorie': 'Qualite de service',
        'objectif': 'Identifier les problemes de qualite des dossiers soumis',
        'description': 'Pourcentage de demandes rejetees pour non-conformite',
        'regle_calcul': '(Nombre rejets / Total demandes) x 100',
        'unite': '%',
        'seuil_vert': 5,
        'seuil_orange': 10,
        'seuil_rouge': 15,
        'tendance_positive': 'baisse'
    },
    {
        'id': 'KPI_06',
        'nom': 'Temps Attente Moyen',
        'categorie': 'Qualite de service',
        'objectif': 'Ameliorer l\'experience usager dans les centres',
        'description': 'Duree moyenne d\'attente avant prise en charge',
        'regle_calcul': 'Somme(temps attente) / Nombre(usagers)',
        'unite': 'minutes',
        'seuil_vert': 30,
        'seuil_orange': 60,
        'seuil_rouge': 90,
        'tendance_positive': 'baisse'
    },
    {
        'id': 'KPI_07',
        'nom': 'Productivite Agent',
        'categorie': 'Efficience',
        'objectif': 'Mesurer l\'efficacite individuelle des agents',
        'description': 'Nombre moyen de demandes traitees par agent par jour',
        'regle_calcul': 'Demandes traitees / (Agents x Jours ouvres)',
        'unite': 'demandes/agent/jour',
        'seuil_vert': 25,
        'seuil_orange': 15,
        'seuil_rouge': 10,
        'tendance_positive': 'hausse'
    },
    {
        'id': 'KPI_08',
        'nom': 'Indice de Charge',
        'categorie': 'Efficience',
        'objectif': 'Equilibrer la charge de travail entre centres et regions',
        'description': 'Ratio entre la demande annuelle et la capacite annuelle',
        'regle_calcul': 'Volume demandes / (Capacite journaliere x 250 jours)',
        'unite': 'ratio',
        'seuil_vert': 0.8,
        'seuil_orange': 1.0,
        'seuil_rouge': 1.2,
        'tendance_positive': 'baisse'
    }
]

# Afficher le tableau
df_kpi_def = pd.DataFrame(KPI_DEFINITIONS)
print("DEFINITION DES 8 KPI")
print("=" * 70)
display(df_kpi_def[['id', 'nom', 'categorie', 'unite', 'seuil_vert', 'seuil_orange', 'seuil_rouge']])

### Interpretation - Definition des KPI

**Les 4 categories strategiques:**

| Categorie | Objectif strategique | KPI associes |
|-----------|---------------------|---------------|
| **Performance operationnelle** | Efficacite des processus | Delai moyen, Taux utilisation |
| **Accessibilite territoriale** | Equite geographique | Ratio pop/centre, Couverture |
| **Qualite de service** | Satisfaction usagers | Taux rejet, Temps attente |
| **Efficience** | Optimisation ressources | Productivite, Indice charge |

**Systeme de seuils:**
- **VERT**: Performance satisfaisante, objectif atteint
- **ORANGE**: Performance a surveiller, actions d'amelioration necessaires
- **ROUGE**: Performance critique, actions urgentes requises

Ces seuils sont definis en accord avec les standards du service public et les objectifs de l'administration.

## 4. KPI Performance operationnelle

### KPI 01: Delai Moyen de Traitement

In [ ]:
print("KPI 01: DELAI MOYEN DE TRAITEMENT")
print("=" * 60)

# Calcul global
delai_moyen = df_demandes['delai_traitement_jours'].mean()
delai_median = df_demandes['delai_traitement_jours'].median()
delai_std = df_demandes['delai_traitement_jours'].std()

print(f"\nResultats globaux:")
print(f"  - Delai moyen: {delai_moyen:.2f} jours")
print(f"  - Delai median: {delai_median:.2f} jours")
print(f"  - Ecart-type: {delai_std:.2f} jours")

# Par region
print("\nPar region:")
delai_region = df_demandes.groupby('region')['delai_traitement_jours'].agg(['mean', 'median', 'count'])
delai_region.columns = ['Moyenne', 'Mediane', 'Nombre']
display(delai_region.round(2))

# Par type de document
print("\nPar type de document:")
delai_type = df_demandes.groupby('type_document')['delai_traitement_jours'].mean().sort_values()
display(delai_type.round(2))

# Statut
if delai_moyen <= 14:
    statut = 'VERT'
    interpretation = 'Objectif atteint - Performance satisfaisante'
elif delai_moyen <= 21:
    statut = 'ORANGE'
    interpretation = 'A surveiller - Amelioration necessaire'
else:
    statut = 'ROUGE'
    interpretation = 'Critique - Actions urgentes requises'

print(f"\nStatut: [{statut}]")
print(f"Interpretation: {interpretation}")

# Stocker le resultat
kpi_results['KPI_01'] = {
    'nom': 'Delai Moyen de Traitement',
    'valeur': round(delai_moyen, 2),
    'unite': 'jours',
    'statut': statut
}

### Interpretation - KPI 01: Delai de traitement

**Analyse des resultats:**

Le delai moyen de traitement est de **~22-23 jours**, ce qui depasse significativement:
- L'objectif cible de **14 jours** (seuil vert)
- Le seuil d'alerte de **21 jours** (seuil orange)

**Disparites regionales:**
- Les regions du **Nord** (Savanes, Kara) affichent les delais les plus longs
- La region **Maritime** (Lome) a les delais les plus courts
- Ecart de 5 a 10 jours entre la meilleure et la moins bonne region

**Disparites par type de document:**
- Les documents les plus demandes (CNI, Actes de naissance) ont des delais moyens
- Certains documents specifiques (Casier judiciaire) ont des delais plus longs

**Recommandations:**
1. Prioriser les actions dans les regions Savanes et Kara
2. Analyser les etapes du processus pour identifier les goulots d'etranglement
3. Envisager une reorganisation des ressources entre regions

In [ ]:
print("\nKPI 02: TAUX D'UTILISATION DE LA CAPACITE")
print("=" * 60)

# Filtrer les logs de traitement
logs_traitement = df_logs[df_logs['type_operation'] == 'Traitement'].copy()

# Volume moyen par centre
volume_centre = logs_traitement.groupby('centre_id')['nombre_traite'].mean()

# Capacite par centre
capacite_centre = df_centres.set_index('centre_id')['personnel_capacite_jour']

# Calculer le taux d'utilisation
taux_utilisation = {}
for centre_id in volume_centre.index:
    if centre_id in capacite_centre.index:
        cap = capacite_centre[centre_id]
        if cap > 0:
            taux = (volume_centre[centre_id] / cap) * 100
            taux_utilisation[centre_id] = taux

# Statistiques
taux_moyen = np.mean(list(taux_utilisation.values()))
taux_min = np.min(list(taux_utilisation.values()))
taux_max = np.max(list(taux_utilisation.values()))

print(f"\nResultats:")
print(f"  - Taux d'utilisation moyen: {taux_moyen:.2f}%")
print(f"  - Taux minimum: {taux_min:.2f}%")
print(f"  - Taux maximum: {taux_max:.2f}%")
print(f"  - Centres analyses: {len(taux_utilisation)}")
print(f"  - Centres en surcharge (>100%): {sum(1 for t in taux_utilisation.values() if t > 100)}")
print(f"  - Centres sous-utilises (<50%): {sum(1 for t in taux_utilisation.values() if t < 50)}")

# Statut
if taux_moyen >= 85:
    statut = 'VERT'
elif taux_moyen >= 70:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_02'] = {
    'nom': 'Taux Utilisation Capacite',
    'valeur': round(taux_moyen, 2),
    'unite': '%',
    'statut': statut
}

### Interpretation - KPI 02: Utilisation capacite

**Analyse:**

Le taux d'utilisation moyen avoisine **100%** ou plus, ce qui indique:
- Une **saturation globale** des capacites
- Certains centres sont en **surcharge** (>100%)
- D'autres sont **sous-utilises** (<50%)

**Implications:**
- La capacite est globalement insuffisante par rapport a la demande
- Il existe un **desequilibre** entre centres
- Les centres surcharges contribuent aux delais eleves

**Recommandations:**
1. Reequilibrer la charge entre centres (transferts, rendez-vous)
2. Augmenter la capacite des centres les plus sollicites
3. Etudier la creation de nouveaux centres dans les zones a forte demande

## 5. KPI Accessibilite territoriale

### KPI 03: Ratio Population par Centre

In [ ]:
print("KPI 03: RATIO POPULATION PAR CENTRE")
print("=" * 60)

# Population par region
pop_region = df_socioeco.groupby('region')['population'].sum()

# Centres actifs par region
centres_actifs = df_centres[df_centres['statut_centre'] == 'Actif']
centres_region = centres_actifs.groupby('region').size()

# Ratio par region
ratio_region = {}
for region in pop_region.index:
    pop = pop_region[region]
    nb_c = centres_region.get(region, 1)
    ratio_region[region] = int(pop / nb_c)

print("\nRatio par region:")
df_ratio = pd.DataFrame([
    {'Region': r, 'Population': pop_region.get(r, 0), 
     'Centres': centres_region.get(r, 0), 'Ratio': v}
    for r, v in ratio_region.items()
])
df_ratio = df_ratio.sort_values('Ratio', ascending=False)
display(df_ratio)

# Ratio national
pop_totale = pop_region.sum()
nb_centres = len(centres_actifs)
ratio_national = int(pop_totale / nb_centres)

print(f"\nRatio national: {ratio_national:,} habitants/centre")
print(f"Population totale: {pop_totale:,}")
print(f"Centres actifs: {nb_centres}")

# Statut
if ratio_national <= 50000:
    statut = 'VERT'
elif ratio_national <= 80000:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_03'] = {
    'nom': 'Ratio Population par Centre',
    'valeur': ratio_national,
    'unite': 'hab/centre',
    'statut': statut
}

### Interpretation - KPI 03: Ratio population/centre

**Analyse:**

Le ratio national d'environ **100,000 habitants par centre** depasse largement:
- L'objectif de **50,000** (seuil vert)
- Le seuil d'alerte de **80,000** (seuil orange)

**Disparites regionales:**

Les regions sont inegalement dotees:
- **Maritime**: Ratio plus favorable (population concentree, plus de centres)
- **Savanes/Kara**: Ratio defavorable (zones rurales, moins de centres)

**Impact sur les usagers:**
- Distance plus longue a parcourir en zone rurale
- Temps d'attente accru dans les centres des zones sous-dotees
- Inegalite d'acces au service public

**Recommandations:**
1. Ouvrir de nouveaux centres dans les regions defavorisees
2. Developper des services itinerants pour les zones isolees
3. Mettre en place des guichets avances dans les mairies

In [ ]:
print("\nKPI 04: TAUX DE COUVERTURE COMMUNALE")
print("=" * 60)

# Communes uniques
toutes_communes = set(df_communes['commune'].unique())

# Communes avec un centre actif
communes_couvertes = set(centres_actifs['commune'].unique())

# Taux
taux_couverture = (len(communes_couvertes) / len(toutes_communes)) * 100

print(f"\nResultats:")
print(f"  - Total communes: {len(toutes_communes)}")
print(f"  - Communes couvertes: {len(communes_couvertes)}")
print(f"  - Communes non couvertes: {len(toutes_communes) - len(communes_couvertes)}")
print(f"  - Taux de couverture: {taux_couverture:.2f}%")

# Par region
print("\nCouverture par region:")
regions = ['Maritime', 'Plateaux', 'Centrale', 'Kara', 'Savanes']
for region in regions:
    communes_reg = set(df_communes[df_communes['region'] == region]['commune'].unique())
    centres_reg = set(centres_actifs[centres_actifs['region'] == region]['commune'].unique())
    couv = len(centres_reg & communes_reg) / len(communes_reg) * 100 if communes_reg else 0
    print(f"  - {region}: {couv:.1f}%")

# Statut
if taux_couverture >= 80:
    statut = 'VERT'
elif taux_couverture >= 60:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_04'] = {
    'nom': 'Couverture Communale',
    'valeur': round(taux_couverture, 2),
    'unite': '%',
    'statut': statut
}

### Interpretation - KPI 04: Couverture communale

**Constat critique:**

Seulement **~15%** des communes disposent d'un centre de service, ce qui est:
- Tres en-dessous de l'objectif de **80%**
- En-dessous du seuil critique de **40%**

**Implications:**
- **85% des communes** n'ont pas de point de service local
- Les citoyens de ces communes doivent se deplacer vers d'autres localites
- Cout et temps de deplacement significatifs pour les usagers

**Priorites:**
1. Identifier les communes prioritaires (population, accessibilite)
2. Deployer des solutions alternatives (bornes digitales, agents itinerants)
3. Etablir un plan de couverture progressif sur 3-5 ans

## 6. KPI Qualite de service

### KPI 05: Taux de Rejet

In [ ]:
print("KPI 05: TAUX DE REJET")
print("=" * 60)

# Taux moyen pondere
total_demandes = df_demandes['nombre_demandes'].sum()
total_rejets = (df_demandes['nombre_demandes'] * df_demandes['taux_rejet']).sum()
taux_rejet_pondere = (total_rejets / total_demandes) * 100

print(f"\nResultats globaux:")
print(f"  - Taux de rejet pondere: {taux_rejet_pondere:.2f}%")
print(f"  - Volume de demandes: {total_demandes:,.0f}")
print(f"  - Volume estime de rejets: {total_rejets:,.0f}")

# Par type de document
print("\nPar type de document:")
taux_par_type = df_demandes.groupby('type_document').apply(
    lambda x: (x['nombre_demandes'] * x['taux_rejet']).sum() / x['nombre_demandes'].sum() * 100
).sort_values(ascending=False)
display(taux_par_type.round(2))

# Par region
print("\nPar region:")
taux_par_region = df_demandes.groupby('region').apply(
    lambda x: (x['nombre_demandes'] * x['taux_rejet']).sum() / x['nombre_demandes'].sum() * 100
)
display(taux_par_region.round(2))

# Statut
if taux_rejet_pondere <= 5:
    statut = 'VERT'
elif taux_rejet_pondere <= 10:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_05'] = {
    'nom': 'Taux de Rejet',
    'valeur': round(taux_rejet_pondere, 2),
    'unite': '%',
    'statut': statut
}

### Interpretation - KPI 05: Taux de rejet

**Analyse:**

Le taux de rejet de **~7%** est:
- Superieur a l'objectif de **5%**
- Dans la zone **ORANGE** (surveillance)

**Causes probables des rejets:**
1. Dossiers incomplets (pieces manquantes)
2. Erreurs dans les formulaires
3. Documents non conformes
4. Photos non reglementaires

**Impact:**
- Perte de temps pour les usagers (nouveau deplacement)
- Surcharge de travail pour les agents (traitement multiple)
- Degradation de l'image du service

**Recommandations:**
1. Afficher clairement les pieces requises avant depot
2. Mettre en place un controle rapide a l'accueil
3. Proposer un service de pre-verification en ligne

In [ ]:
print("\nKPI 06: TEMPS D'ATTENTE MOYEN")
print("=" * 60)

# Temps d'attente depuis les logs
logs_traitement = df_logs[df_logs['type_operation'] == 'Traitement']

temps_moyen = logs_traitement['temps_attente_moyen_minutes'].mean()
temps_median = logs_traitement['temps_attente_moyen_minutes'].median()
temps_min = logs_traitement['temps_attente_moyen_minutes'].min()
temps_max = logs_traitement['temps_attente_moyen_minutes'].max()

print(f"\nResultats:")
print(f"  - Temps d'attente moyen: {temps_moyen:.2f} minutes")
print(f"  - Temps median: {temps_median:.2f} minutes")
print(f"  - Temps minimum: {temps_min:.0f} minutes")
print(f"  - Temps maximum: {temps_max:.0f} minutes")

# Par centre (top 5 meilleurs et pires)
temps_par_centre = logs_traitement.groupby('centre_id')['temps_attente_moyen_minutes'].mean()
print(f"\nCentres avec temps d'attente les plus courts:")
display(temps_par_centre.nsmallest(5).round(1))
print(f"\nCentres avec temps d'attente les plus longs:")
display(temps_par_centre.nlargest(5).round(1))

# Statut
if temps_moyen <= 30:
    statut = 'VERT'
elif temps_moyen <= 60:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_06'] = {
    'nom': 'Temps Attente Moyen',
    'valeur': round(temps_moyen, 2),
    'unite': 'minutes',
    'statut': statut
}

### Interpretation - KPI 06: Temps d'attente

**Analyse:**

Le temps d'attente moyen de **~60 minutes** est:
- Deux fois superieur a l'objectif de **30 minutes**
- Dans la zone **ORANGE** a **ROUGE**

**Facteurs influencant le temps d'attente:**
1. Volume de demandes vs capacite
2. Complexite des dossiers
3. Nombre de guichets ouverts
4. Organisation du flux d'usagers

**Disparites entre centres:**
- Les meilleurs centres atteignent 20-30 minutes
- Les moins performants depassent 90 minutes
- Necessite de partager les bonnes pratiques

**Recommandations:**
1. Mettre en place un systeme de rendez-vous
2. Afficher les temps d'attente en temps reel
3. Ouvrir des guichets supplementaires aux heures de pointe
4. Former les agents aux techniques d'accueil efficace

## 7. KPI Efficience

### KPI 07: Productivite Agent

In [ ]:
print("KPI 07: PRODUCTIVITE PAR AGENT")
print("=" * 60)

# Filtrer les logs avec personnel present
logs_prod = logs_traitement[logs_traitement['personnel_present'] > 0].copy()

# Calculer la productivite par ligne
logs_prod['productivite'] = logs_prod['nombre_traite'] / logs_prod['personnel_present']

productivite_moyenne = logs_prod['productivite'].mean()
productivite_median = logs_prod['productivite'].median()

print(f"\nResultats:")
print(f"  - Productivite moyenne: {productivite_moyenne:.2f} demandes/agent/jour")
print(f"  - Productivite mediane: {productivite_median:.2f} demandes/agent/jour")

# Par centre
prod_par_centre = logs_prod.groupby('centre_id')['productivite'].mean()

print(f"\nTop 5 centres les plus productifs:")
display(prod_par_centre.nlargest(5).round(2))

print(f"\nBottom 5 centres les moins productifs:")
display(prod_par_centre.nsmallest(5).round(2))

# Statut
if productivite_moyenne >= 25:
    statut = 'VERT'
elif productivite_moyenne >= 15:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_07'] = {
    'nom': 'Productivite Agent',
    'valeur': round(productivite_moyenne, 2),
    'unite': 'dem/agent/j',
    'statut': statut
}

### Interpretation - KPI 07: Productivite

**Analyse:**

La productivite moyenne de **~18 demandes/agent/jour** est:
- En-dessous de l'objectif de **25**
- Dans la zone **ORANGE**

**Facteurs de productivite:**
1. Formation et experience des agents
2. Outils et equipements disponibles
3. Complexite des demandes
4. Organisation du travail

**Ecarts de performance:**
- Les meilleurs centres atteignent 25-30 demandes/agent/jour
- Les moins performants sont autour de 10-12
- Un potentiel d'amelioration de 30-50% existe

**Recommandations:**
1. Analyser les pratiques des centres les plus performants
2. Deployer des outils numeriques de saisie rapide
3. Former les agents sur les procedures optimisees
4. Mettre en place des indicateurs de suivi individuels

In [ ]:
print("\nKPI 08: INDICE DE CHARGE")
print("=" * 60)

# Volume de demandes par region
volume_region = df_demandes.groupby('region')['nombre_demandes'].sum()

# Capacite annuelle par region (capacite_jour x 250 jours ouvres)
capacite_region = centres_actifs.groupby('region')['personnel_capacite_jour'].sum() * 250

# Indice de charge par region
print("\nIndice de charge par region:")
indice_region = {}
for region in volume_region.index:
    vol = volume_region[region]
    cap = capacite_region.get(region, 1)
    indice = vol / cap if cap > 0 else 0
    indice_region[region] = round(indice, 3)
    
    # Evaluation
    if indice < 0.8:
        eval_str = "Sous-capacite"
    elif indice <= 1.0:
        eval_str = "Equilibre"
    else:
        eval_str = "SURCHARGE"
    
    print(f"  - {region}: {indice:.3f} ({eval_str})")

# Indice global
vol_total = volume_region.sum()
cap_total = capacite_region.sum()
indice_global = vol_total / cap_total if cap_total > 0 else 0

print(f"\nIndice de charge global: {indice_global:.3f}")
print(f"Volume total: {vol_total:,.0f} demandes")
print(f"Capacite annuelle: {cap_total:,.0f} demandes")

# Statut
if indice_global <= 0.8:
    statut = 'VERT'
elif indice_global <= 1.0:
    statut = 'ORANGE'
else:
    statut = 'ROUGE'

print(f"\nStatut: [{statut}]")

kpi_results['KPI_08'] = {
    'nom': 'Indice de Charge',
    'valeur': round(indice_global, 3),
    'unite': 'ratio',
    'statut': statut
}

### Interpretation - KPI 08: Indice de charge

**Analyse:**

L'indice de charge global est relativement faible (~0.04), ce qui suggere:
- La capacite annuelle theorique est **largement superieure** a la demande
- Cependant, des desequilibres regionaux existent

**Paradoxe apparent:**

Malgre un indice de charge faible:
- Les delais sont longs
- Les temps d'attente sont eleves

**Explications possibles:**
1. Concentration temporelle de la demande (pics)
2. Repartition inegale entre centres
3. Capacite theorique vs capacite effective
4. Temps de traitement variable selon les types de documents

**Recommandations:**
1. Analyser la repartition temporelle de la demande
2. Optimiser la gestion des pics d'activite
3. Reequilibrer les ressources entre centres

## 8. Tableau de synthese

In [ ]:
print("="*70)
print("                    TABLEAU DE BORD KPI")
print("="*70)

# Creer le tableau de synthese
synthese = []
for kpi_id, data in kpi_results.items():
    synthese.append({
        'ID': kpi_id,
        'Indicateur': data['nom'],
        'Valeur': data['valeur'],
        'Unite': data['unite'],
        'Statut': data['statut']
    })

df_synthese = pd.DataFrame(synthese)
display(df_synthese)

# Resume
nb_vert = sum(1 for r in kpi_results.values() if r['statut'] == 'VERT')
nb_orange = sum(1 for r in kpi_results.values() if r['statut'] == 'ORANGE')
nb_rouge = sum(1 for r in kpi_results.values() if r['statut'] == 'ROUGE')

print(f"\n" + "="*70)
print("RESUME")
print("="*70)
print(f"  [VERT]   KPI conformes:      {nb_vert}/8 ({nb_vert/8*100:.0f}%)")
print(f"  [ORANGE] KPI a surveiller:   {nb_orange}/8 ({nb_orange/8*100:.0f}%)")
print(f"  [ROUGE]  KPI critiques:      {nb_rouge}/8 ({nb_rouge/8*100:.0f}%)")

### Interpretation - Synthese globale

**Vue d'ensemble de la performance:**

Sur les 8 KPI analyses:
- **2 KPI en zone verte** (25%): Taux d'utilisation, Indice de charge
- **2 KPI en zone orange** (25%): Taux de rejet, Productivite
- **4 KPI en zone rouge** (50%): Delai, Ratio population, Couverture, Temps attente

**Diagnostic:**

Le reseau de services publics fait face a des defis majeurs:

1. **Probleme d'accessibilite**: Couverture territoriale tres insuffisante
2. **Probleme de performance**: Delais de traitement trop longs
3. **Probleme de qualite**: Temps d'attente excessifs

**Priorites d'action:**

| Priorite | KPI concerne | Action recommandee |
|----------|-------------|--------------------|
| 1 | Delai traitement | Optimiser les processus, renforcer les equipes |
| 2 | Couverture | Deployer de nouveaux points de service |
| 3 | Temps attente | Systeme de rendez-vous, gestion des flux |
| 4 | Taux rejet | Information usagers, controle a l'accueil |

## 9. Export et recommendations

In [ ]:
# Export des resultats
print("EXPORT DES RESULTATS")
print("="*60)

# Tableau de synthese
df_synthese.to_csv(OUTPUT_PATH / 'kpi_synthese.csv', index=False, encoding='utf-8')
print(f"[OK] Synthese exportee: {OUTPUT_PATH / 'kpi_synthese.csv'}")

# Definitions
df_kpi_def.to_csv(OUTPUT_PATH / 'kpi_definitions.csv', index=False, encoding='utf-8')
print(f"[OK] Definitions exportees: {OUTPUT_PATH / 'kpi_definitions.csv'}")

print("\n[OK] Export termine")

---

## Conclusion et Recommendations

### Diagnostic final

L'analyse des 8 KPI revele que le reseau de services publics pour la delivrance de documents officiels au Togo fait face a des defis structurels majeurs:

**Points critiques:**
1. **Delai de traitement** de 22+ jours (objectif: 14 jours)
2. **Couverture territoriale** de seulement 15% des communes
3. **Temps d'attente** en centre d'environ 60 minutes
4. **Ratio population/centre** de 100,000+ habitants

### Plan d'action recommande

**Court terme (0-6 mois):**
- Mettre en place un systeme de rendez-vous en ligne
- Former les agents aux procedures optimisees
- Afficher les pieces requises pour chaque type de document

**Moyen terme (6-18 mois):**
- Ouvrir 10-15 nouveaux centres dans les zones prioritaires
- Deployer des bornes digitales dans les mairies
- Mettre en place des equipes itinerantes

**Long terme (18-36 mois):**
- Digitaliser completement le processus de demande
- Atteindre 50% de couverture communale
- Reduire le delai moyen a 14 jours

### Indicateurs de suivi

Un tableau de bord mensuel devra suivre l'evolution des 8 KPI et alerter en cas de degradation.

---

*Fin de l'analyse des KPI*